# PIC_SUB_01: outcome-blind feasibility check for replicating the admission subphenotypes in PIC

**Run this notebook on your own computer only.** PIC (Paediatric Intensive Care database, PhysioNet v1.1.0) is credentialed
data. It must not be uploaded to any online service, including chat assistants (PhysioNet guidance of 24 September 2025). The
notebook writes **aggregate outputs only** (counts, percentages, medians and dictionary labels), and every count from 1 to 4 is
shown as "<5". Only the files in `./output_pic_sub01/` may be shared, after a look at their content.

**Why.** The registered study (OSF osf.io/ygtr2) lists replication in the PIC database as an exploratory analysis. The analysis
plan (decision P15) fixes the order: first an aggregate count of eligible children; replication only if at least 200 are found;
the Wuhan model is then applied to them without refitting. This notebook makes the count and the checks needed to fix the PIC
definitions (diagnosis codes, laboratory items, units) **before** any child is assigned to a subphenotype. It computes nothing by
subphenotype, and the only outcome it counts is the overall number of in-hospital deaths, which is needed to judge precision.

**How to run.** Put this notebook in the folder with the PIC CSV files (`.csv` or `.csv.gz`, file and column names in any case),
or set `PIC_DIR` below, and choose *Restart Kernel and Run All Cells*. LABEVENTS and CHARTEVENTS are read in chunks, so a run takes
a few minutes. Requirements: Python 3.9 or later, pandas 2.0 or later and NumPy (the `ae_subpheno` environment works).

**Outputs** in `./output_pic_sub01/`: `summary.md` (paste it back, or send the whole folder) and the tables it lists.

In [ ]:
# Setup and parameters.
import re, sys, json, time, gzip, platform, warnings
from pathlib import Path
import numpy as np
import pandas as pd

sys.dont_write_bytecode = True
warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 70)

PIC_DIR = None                  # folder with the PIC CSV files; None = the folder of this notebook
AGE_MIN_DAYS = 28               # neonates are excluded, as in Wuhan
AGE_MAX_YEARS = 18
LAB_WINDOW_H = (-24, 24)        # first value from 24 h before to 24 h after admission (Wuhan definition)
VALIDATOR_WINDOW_H = (-24, 48)  # cytokines and ferritin (Wuhan definition)
VITAL_WINDOW_H = (0, 24)        # first charted value within 24 h after admission (Wuhan definition)
MIN_TESTS = 11                  # of the 22 blood tests (Wuhan clustering population)
GATE_N = 200                    # registered threshold for going ahead with the replication (analysis plan, decision P15)
CHUNK = 500_000
DX_STATUS_OVERRIDE = {}         # e.g. {'G93.400': 'exclude'} after review of 02_dx_codes.csv
LAB_ITEMID_OVERRIDE = {}        # e.g. {'hs_crp': ['5xxx']} after review of 05_lab_items.csv
VITAL_ITEMID_OVERRIDE = {}      # e.g. {'heart_rate': ['1003']} after review of 06_vital_items.csv

HERE = Path.cwd()
SRC = Path(PIC_DIR) if PIC_DIR else HERE
OUT = HERE / 'output_pic_sub01'
OUT.mkdir(exist_ok=True)
T0 = time.time()
FIXTURE = (SRC / 'SYNTHETIC_TEST_FIXTURE.txt').exists()
if FIXTURE:
    print('#' * 100 + '\nTEST RUN ON A SYNTHETIC FIXTURE: every number below is meaningless and must never be reported.\n' + '#' * 100)
print('python', platform.python_version(), '| pandas', pd.__version__, '| numpy', np.__version__)


def sup(n):
    """Small-cell suppression: counts from 1 to 4 are shown as '<5'."""
    if n is None or (isinstance(n, float) and np.isnan(n)):
        return None
    n = int(n)
    return '<5' if 0 < n < 5 else n


def pct(k, n, digits=1):
    """Percentage; blank when the numerator is 1-4 or the denominator is 0."""
    k = int(k)
    if not n or 0 < k < 5:
        return None
    return round(100 * k / n, digits)


def q(s):
    """Median (25th-75th percentile) as text; blank for fewer than 5 values."""
    s = pd.Series(s, dtype=float).dropna()
    if len(s) < 5:
        return None
    f = lambda x: f'{x:.3g}'
    return f'{f(s.median())} ({f(s.quantile(.25))}-{f(s.quantile(.75))})'


def save(df, name):
    df.to_csv(OUT / name, index=False, encoding='utf-8-sig')
    return df

In [ ]:
# Wuhan reference values (aggregate): 695 clustered children of AE_freeze_v1.4, first value in the measurement window. Used only to
# check that PIC items and units are on the Wuhan scale; the subphenotype model itself is not in this notebook.
WUHAN = {"neutrophils_abs": {"unit": "10^9/L", "modal_reference_limit": 5.8, "pct_measured_clustered": 94.0, "median": 5.09, "p25": 2.63, "p75": 8.73}, "wbc": {"unit": "10^9/L", "modal_reference_limit": 11.9, "pct_measured_clustered": 94.1, "median": 8.03, "p25": 5.23, "p75": 11.84}, "lymphocytes_pct": {"unit": "%", "modal_reference_limit": 23.0, "pct_measured_clustered": 86.2, "median": 23.9, "p25": 12.7, "p75": 40.25}, "platelets": {"unit": "10^9/L", "modal_reference_limit": 100.0, "pct_measured_clustered": 94.0, "median": 250.0, "p25": 169.0, "p75": 331.0}, "hemoglobin": {"unit": "g/L", "modal_reference_limit": 110.0, "pct_measured_clustered": 94.0, "median": 116.0, "p25": 106.0, "p75": 126.0}, "hs_crp": {"unit": "mg/L", "modal_reference_limit": 3.0, "pct_measured_clustered": 84.3, "median": 5.635, "p25": 0.9825, "p75": 21.6}, "procalcitonin": {"unit": "ng/mL", "modal_reference_limit": 0.05, "pct_measured_clustered": 89.5, "median": 0.43, "p25": 0.12, "p75": 5.215}, "alt": {"unit": "U/L", "modal_reference_limit": 30.0, "pct_measured_clustered": 99.4, "median": 23.0, "p25": 15.0, "p75": 39.0}, "ast": {"unit": "U/L", "modal_reference_limit": 44.0, "pct_measured_clustered": 99.4, "median": 44.0, "p25": 30.0, "p75": 73.5}, "ldh": {"unit": "U/L", "modal_reference_limit": 300.0, "pct_measured_clustered": 98.4, "median": 296.0, "p25": 240.8, "p75": 474.8}, "ck": {"unit": "U/L", "modal_reference_limit": 250.0, "pct_measured_clustered": 98.4, "median": 124.5, "p25": 72.0, "p75": 241.5}, "bilirubin_total": {"unit": "umol/L", "modal_reference_limit": 19.0, "pct_measured_clustered": 99.4, "median": 6.6, "p25": 4.4, "p75": 9.9}, "albumin": {"unit": "g/L", "modal_reference_limit": 39.0, "pct_measured_clustered": 99.4, "median": 41.5, "p25": 37.6, "p75": 44.8}, "creatinine": {"unit": "umol/L", "modal_reference_limit": 66.0, "pct_measured_clustered": 99.4, "median": 31.1, "p25": 24.85, "p75": 40.7}, "urea": {"unit": "mmol/L", "modal_reference_limit": 7.1, "pct_measured_clustered": 99.4, "median": 3.9, "p25": 2.8, "p75": 5.395}, "sodium": {"unit": "mmol/L", "modal_reference_limit": 135.0, "pct_measured_clustered": 99.1, "median": 136.9, "p25": 134.2, "p75": 139.3}, "ammonia": {"unit": "umol/L", "modal_reference_limit": 72.0, "pct_measured_clustered": 74.2, "median": 23.0, "p25": 16.0, "p75": 42.0}, "lactate": {"unit": "mmol/L", "modal_reference_limit": 2.22, "pct_measured_clustered": 67.9, "median": 2.225, "p25": 1.62, "p75": 3.3}, "inr": {"unit": "ratio", "modal_reference_limit": 1.16, "pct_measured_clustered": 80.7, "median": 1.05, "p25": 0.97, "p75": 1.2}, "aptt": {"unit": "s", "modal_reference_limit": 39.0, "pct_measured_clustered": 80.9, "median": 32.7, "p25": 27.82, "p75": 39.88}, "fibrinogen": {"unit": "g/L", "modal_reference_limit": 2.0, "pct_measured_clustered": 80.9, "median": 2.29, "p25": 1.77, "p75": 2.955}, "d_dimer": {"unit": "mg/L FEU", "modal_reference_limit": 0.55, "pct_measured_clustered": 67.3, "median": 0.6, "p25": 0.29, "p75": 2.24}, "il6": {"unit": "pg/mL", "pct_measured_clustered": 61.9, "median": 16.52, "p25": 6.825, "p75": 64.94}, "il10": {"unit": "pg/mL", "pct_measured_clustered": 61.9, "median": 8.255, "p25": 4.525, "p75": 25.09}, "tnf_alpha": {"unit": "pg/mL", "pct_measured_clustered": 61.7, "median": 2.9, "p25": 1.91, "p75": 4.43}, "ifn_gamma": {"unit": "pg/mL", "pct_measured_clustered": 31.2, "median": 2.72, "p25": 2.07, "p75": 4.13}, "ferritin": {"unit": "ng/mL", "pct_measured_clustered": 54.7, "median": 164.9, "p25": 85.9, "p75": 479.0}, "temp_c": {"pct_measured_clustered": 99.3, "median": 36.8, "p25": 36.5, "p75": 37.6}, "heart_rate": {"pct_measured_clustered": 88.3, "median": 121.0, "p25": 101.0, "p75": 142.0}, "resp_rate": {"pct_measured_clustered": 88.2, "median": 27.0, "p25": 23.0, "p75": 34.0}}
LAB22 = ['neutrophils_abs', 'wbc', 'lymphocytes_pct', 'platelets', 'hemoglobin', 'hs_crp', 'procalcitonin', 'alt', 'ast', 'ldh', 'ck',
         'bilirubin_total', 'albumin', 'creatinine', 'urea', 'sodium', 'ammonia', 'lactate', 'inr', 'aptt', 'fibrinogen', 'd_dimer']
VALIDATORS = ['il6', 'il10', 'tnf_alpha', 'ifn_gamma', 'ferritin']
MODEL24 = [c for c in LAB22 if c != 'wbc'] + ['temp_c', 'heart_rate', 'resp_rate']     # the 24 clustering indicators
LABEL = {'neutrophils_abs': 'Neutrophil count', 'wbc': 'White cell count', 'lymphocytes_pct': 'Lymphocyte percentage', 'platelets': 'Platelets',
         'hemoglobin': 'Hemoglobin', 'hs_crp': 'CRP', 'procalcitonin': 'Procalcitonin', 'alt': 'ALT', 'ast': 'AST', 'ldh': 'LDH', 'ck': 'CK',
         'bilirubin_total': 'Total bilirubin', 'albumin': 'Albumin', 'creatinine': 'Creatinine', 'urea': 'Urea', 'sodium': 'Sodium',
         'ammonia': 'Ammonia', 'lactate': 'Lactate', 'inr': 'INR', 'aptt': 'APTT', 'fibrinogen': 'Fibrinogen', 'd_dimer': 'D-dimer',
         'il6': 'IL-6', 'il10': 'IL-10', 'tnf_alpha': 'TNF-alpha', 'ifn_gamma': 'IFN-gamma', 'ferritin': 'Ferritin',
         'temp_c': 'Temperature', 'heart_rate': 'Heart rate', 'resp_rate': 'Respiratory rate'}

In [ ]:
# Files and columns.
ALIASES = {'D_DIAGNOSES_ICD': 'D_ICD_DIAGNOSES'}
FILES = {}
for p in sorted(SRC.iterdir()):
    m = re.match(r'^(.*?)\.csv(\.gz)?$', p.name, flags=re.I)
    if m:
        t = m.group(1).upper()
        FILES[ALIASES.get(t, t)] = p


def read(table, usecols=None, chunksize=None):
    """Read a PIC table as text with upper-case column names; usecols are given in upper case (missing ones are skipped)."""
    p = FILES[table]
    head = pd.read_csv(p, nrows=0)
    cmap = {c: c.strip().upper() for c in head.columns}
    cols = [c for c in head.columns if usecols is None or cmap[c] in usecols]
    kw = dict(usecols=cols, dtype=str, keep_default_na=False, na_values=[''], encoding='utf-8')
    if chunksize:
        return (ch.rename(columns=cmap) for ch in pd.read_csv(p, chunksize=chunksize, **kw))
    return pd.read_csv(p, **kw).rename(columns=cmap)


def count_rows(p):
    opener = gzip.open if p.name.lower().endswith('.gz') else open
    with opener(p, 'rb') as f:
        return max(sum(buf.count(b'\n') for buf in iter(lambda: f.read(1 << 24), b'')) - 1, 0)


def pick(cols, *names, contains=None):
    """First column among names; else the first column whose name contains every string in contains."""
    cols = list(cols)
    for n in names:
        if n in cols:
            return n
    if contains:
        for c in cols:
            if all(k in c for k in contains):
                return c
    return None


REQUIRED = ['PATIENTS', 'ADMISSIONS', 'ICUSTAYS', 'DIAGNOSES_ICD', 'D_ICD_DIAGNOSES', 'D_LABITEMS', 'LABEVENTS', 'D_ITEMS', 'CHARTEVENTS']
FILE_ROWS = []
for t in sorted(FILES):
    cols = [c.strip().upper() for c in pd.read_csv(FILES[t], nrows=0).columns]
    FILE_ROWS.append({'table': t, 'file': FILES[t].name, 'table_rows': count_rows(FILES[t]), 'columns': ', '.join(cols)})
FILE_DF = save(pd.DataFrame(FILE_ROWS), '01_files.csv')
print(FILE_DF[['table', 'file', 'table_rows']].to_string(index=False))
missing = [t for t in REQUIRED if t not in FILES]
if missing:
    raise FileNotFoundError(f'Required PIC tables not found in {SRC}: {missing}')

In [ ]:
# Patients, admissions and ICU stays.
def to_dt(s):
    return pd.to_datetime(s, errors='coerce')


PAT = read('PATIENTS', ['SUBJECT_ID', 'GENDER', 'DOB'])
ADM = read('ADMISSIONS', ['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'DIAGNOSIS', 'ICD10_CODE_CN', 'HOSPITAL_EXPIRE_FLAG'])
for c in ['DIAGNOSIS', 'ICD10_CODE_CN', 'DEATHTIME']:
    if c not in ADM:
        ADM[c] = np.nan
ICU = read('ICUSTAYS', ['HADM_ID', 'ICUSTAY_ID', 'FIRST_CAREUNIT', 'INTIME', 'OUTTIME'])
PAT['DOB'] = to_dt(PAT.DOB)
for c in ['ADMITTIME', 'DISCHTIME', 'DEATHTIME']:
    ADM[c] = to_dt(ADM[c])
ICU['INTIME'] = to_dt(ICU.INTIME)
ADM = ADM.merge(PAT, on='SUBJECT_ID', how='left')
ADM['AGE_DAYS'] = (ADM.ADMITTIME - ADM.DOB).dt.total_seconds() / 86400
ADM['DIED'] = pd.to_numeric(ADM.HOSPITAL_EXPIRE_FLAG, errors='coerce').eq(1)
ADM['LOS_DAYS'] = (ADM.DISCHTIME - ADM.ADMITTIME).dt.total_seconds() / 86400
FIRST_ICU = ICU.dropna(subset=['INTIME']).sort_values(['HADM_ID', 'INTIME']).drop_duplicates('HADM_ID').set_index('HADM_ID')
ADM['ICU_IN'] = ADM.HADM_ID.map(FIRST_ICU.INTIME)
ADM['FIRST_UNIT'] = ADM.HADM_ID.map(FIRST_ICU.FIRST_CAREUNIT)
ADM = ADM.drop_duplicates('HADM_ID').set_index('HADM_ID', drop=False)
ADM.index.name = None
AGE_OK = ADM.AGE_DAYS.ge(AGE_MIN_DAYS) & ADM.AGE_DAYS.lt(AGE_MAX_YEARS * 365.25)
print(f'Admissions {len(ADM)}, children {ADM.SUBJECT_ID.nunique()}, aged 28 days to 17 years at admission {int(AGE_OK.sum())}; '
      f'admissions without admission time {int(ADM.ADMITTIME.isna().sum())}, without date of birth {int(ADM.DOB.isna().sum())}')

## Diagnosis codes
The Wuhan cohort was defined by acute encephalopathy labels in the discharge diagnoses: acute necrotizing encephalopathy (ANE),
hemorrhagic shock and encephalopathy syndrome or ASEM, Reye syndrome, AESD, FIRES, MERS, and septic, pertussis-associated,
influenza-associated, toxic and unspecified acute encephalopathy; sequelae did not count. The rules below apply the same label set to
the PIC diagnosis dictionary (Chinese and English titles and the ICD-10 codes). A code is classified only when it names an
encephalopathy, encephalitis or one of the listed syndromes; exclusions are checked first, then the syndromes in the Wuhan order,
then encephalitis (which is not part of the primary definition and is counted as an alternative). Every classified code is listed in
`02_dx_codes.csv` with its counts, for review; `DX_STATUS_OVERRIDE` changes a code's status.

In [ ]:
ENC_TERMS = (r'脑病|encephalopath|脑炎|encephalit|脑脊髓炎|encephalomyel|瑞氏|瑞夷|雷耶|赖氏|Reye|胼胝体压部|splenial|双相.{0,10}惊厥|'
             r'biphasic seizure|发热感染相关.{0,6}癫痫|febrile infection.related epilepsy|FIRES|颅内感染|中枢神经系统感染|CNS infection|'
             r'\bG92|G93\.4|G93\.7|\bG94|\bG04|\bG05|\bA8[3-6]|B00\.4')
DX_RULES = [  # (status, category, pattern); the first match decides
    ('exclude', 'Hypoxic-ischemic or anoxic injury', r'缺氧缺血|缺血缺氧|缺氧性|hypoxic|anoxic|P91\.6'),
    ('exclude', 'Hepatic encephalopathy', r'肝性脑病|肝昏迷|hepatic|K72'),
    ('exclude', 'Bilirubin encephalopathy', r'胆红素|核黄疸|kernicterus|bilirubin|P57'),
    ('exclude', 'Hypertensive encephalopathy or PRES', r'高血压|hypertensive|可逆性后部|posterior reversible|I67\.4'),
    ('exclude', 'Epileptic encephalopathy', r'癫痫性脑病|epileptic encephalopathy|婴儿痉挛|Lennox|Dravet|大田原|Ohtahara|West'),
    ('exclude', 'Metabolic, mitochondrial or nutritional encephalopathy',
     r'线粒体|mitochondria|MELAS|Leigh|代谢性|metabolic|韦尼克|Wernicke|尿毒症|ur(a)?emic|透析|dialysis|低血糖|hypoglyc|高氨|hyperammon'),
    ('exclude', 'Leukoencephalopathy', r'白质脑病|leukoencephalopath'),
    ('exclude', 'Sequelae or chronic condition', r'后遗症|sequela|陈旧|慢性|chronic|静止性|static'),
    ('include', 'ANE', r'坏死性脑病|坏死性脑炎|necroti[sz]ing encephal'),
    ('include', 'HSES or ASEM', r'出血性休克.{0,4}脑病|hemorrhagic shock.{0,10}encephalopathy|HSES'),
    ('include', 'Reye syndrome', r'瑞氏|瑞夷|雷耶|赖氏|Reye|G93\.7'),
    ('include', 'AESD', r'双相.{0,10}惊厥|biphasic seizure|迟发性.{0,6}弥散|AESD'),
    ('include', 'FIRES', r'发热感染相关.{0,6}癫痫|febrile infection.related epilepsy|FIRES'),
    ('include', 'MERS', r'胼胝体压部|splenial'),
    ('include', 'Septic encephalopathy', r'脓毒.{0,6}脑病|败血.{0,6}脑病|感染相关性脑病|sepsis.associated encephalopathy|septic encephalopathy'),
    ('include', 'Pertussis encephalopathy', r'百日咳.{0,6}脑病|pertussis.{0,10}encephalopathy'),
    ('include', 'Influenza-associated encephalopathy', r'流感.{0,6}脑病|流行性感冒.{0,6}脑病|influenza.{0,30}encephalopathy'),
    ('include', 'Toxic encephalopathy', r'中毒性脑病|toxic encephalopathy|\bG92'),
    ('include', 'Acute or unspecified encephalopathy', r'脑病|encephalopath|G93\.4|\bG94'),
    ('encephalitis', 'Encephalitis or CNS infection', r'.'),
]
CATEGORY_ORDER = [c for s, c, _ in DX_RULES if s == 'include']


def classify(text):
    if not isinstance(text, str) or not re.search(ENC_TERMS, text, flags=re.I):
        return None, None
    for status, category, pat in DX_RULES:
        if re.search(pat, text, flags=re.I):
            return status, category
    return None, None


DXD = read('D_ICD_DIAGNOSES')
C_CODE = pick(DXD.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DXD.columns, 'ICD10_CODE', contains=['CODE'])
C_STD = pick([c for c in DXD.columns if c != C_CODE], 'ICD10_CODE', contains=['CODE'])
C_TCN = pick(DXD.columns, 'TITLE_CN', contains=['TITLE', 'CN'])
C_TEN = pick([c for c in DXD.columns if c != C_TCN], 'TITLE', 'LONG_TITLE', contains=['TITLE'])
print('Diagnosis dictionary columns used:', {'code': C_CODE, 'icd10': C_STD, 'title_cn': C_TCN, 'title_en': C_TEN})
DXD = pd.DataFrame({'CODE': DXD[C_CODE], 'ICD10': DXD[C_STD] if C_STD else np.nan, 'TITLE_CN': DXD[C_TCN] if C_TCN else np.nan,
                    'TITLE': DXD[C_TEN] if C_TEN else np.nan}).drop_duplicates('CODE')
DXD['TXT'] = DXD[['CODE', 'ICD10', 'TITLE_CN', 'TITLE']].fillna('').astype(str).agg(' | '.join, axis=1)
DXD[['STATUS', 'CATEGORY']] = pd.DataFrame([classify(t) for t in DXD.TXT], index=DXD.index)
DX = read('DIAGNOSES_ICD')
C_DXCODE = pick(DX.columns, 'ICD10_CODE_CN', contains=['CODE', 'CN']) or pick(DX.columns, 'ICD10_CODE', contains=['CODE'])
DX = DX[['SUBJECT_ID', 'HADM_ID', C_DXCODE]].rename(columns={C_DXCODE: 'CODE'}).dropna(subset=['CODE']).drop_duplicates(['HADM_ID', 'CODE'])
DX = DX.merge(DXD[['CODE', 'STATUS', 'CATEGORY']], on='CODE', how='left')
_nodict = ~DX.CODE.isin(set(DXD.CODE))
if _nodict.any():
    DX.loc[_nodict, ['STATUS', 'CATEGORY']] = pd.DataFrame([classify(c) for c in DX.loc[_nodict, 'CODE']], index=DX.index[_nodict]).values
for code, status in DX_STATUS_OVERRIDE.items():
    DX.loc[DX.CODE.eq(code), 'STATUS'] = status
    DXD.loc[DXD.CODE.eq(code), 'STATUS'] = status
print(f'Diagnosis rows {len(DX)}; codes not in the dictionary {int(_nodict.sum())} rows')

DXC = DX[DX.STATUS.notna()].copy()
INC = DXC[DXC.STATUS.eq('include')]
ADM_INC = set(INC.HADM_ID) & set(ADM.index)
ADM_ENC = set(DXC[DXC.STATUS.eq('encephalitis')].HADM_ID) & set(ADM.index)
_rank = {c: i for i, c in enumerate(CATEGORY_ORDER)}
ADM_CAT = INC.assign(r=INC.CATEGORY.map(_rank)).sort_values('r').drop_duplicates('HADM_ID').set_index('HADM_ID').CATEGORY
ADM_EXCL = DXC[DXC.STATUS.eq('exclude')].groupby('HADM_ID').CATEGORY.agg(lambda s: sorted(set(s)))

# Admission diagnosis (free text and code), an alternative source; segments are classified one by one.
def classify_segments(*texts):
    found = []
    for t in texts:
        if isinstance(t, str):
            for seg in re.split(r'[;；,，、\n]+', t):
                st, cat = classify(seg.strip())
                if st:
                    found.append((st, cat))
    return found


_txt = {h: classify_segments(r.DIAGNOSIS, r.ICD10_CODE_CN) for h, r in ADM[['DIAGNOSIS', 'ICD10_CODE_CN']].iterrows()}
ADM_TXT_INC = {h for h, f in _txt.items() if any(s == 'include' for s, _ in f)}
for h, f in _txt.items():                 # category for admissions found only through the admission diagnosis
    if h not in ADM_CAT.index and any(s == 'include' for s, _ in f):
        ADM_CAT.loc[h] = sorted({c for s, c in f if s == 'include'}, key=_rank.get)[0]

# Code table for review (counts of admissions; all ages and aged 28 days to 17 years).
_age_ok = set(ADM.index[AGE_OK])
code_rows = []
for code, g in DXC.groupby('CODE'):
    d = DXD[DXD.CODE.eq(code)]
    code_rows.append({'code': code, 'icd10': d.ICD10.iloc[0] if len(d) else None, 'title_cn': d.TITLE_CN.iloc[0] if len(d) else None,
                      'title_en': d.TITLE.iloc[0] if len(d) else None, 'status': g.STATUS.iloc[0], 'category': g.CATEGORY.iloc[0],
                      'admissions_all_ages': g.HADM_ID.nunique(), 'admissions_aged_28d_to_17y': len(set(g.HADM_ID) & _age_ok)})
for code, d in DXD[DXD.STATUS.notna() & ~DXD.CODE.isin(set(DXC.CODE))].iterrows():        # classified codes that are never used
    code_rows.append({'code': d.CODE, 'icd10': d.ICD10, 'title_cn': d.TITLE_CN, 'title_en': d.TITLE, 'status': d.STATUS,
                      'category': d.CATEGORY, 'admissions_all_ages': 0, 'admissions_aged_28d_to_17y': 0})
CODES = pd.DataFrame(code_rows, columns=['code', 'icd10', 'title_cn', 'title_en', 'status', 'category', 'admissions_all_ages',
                                         'admissions_aged_28d_to_17y'])
CODES = CODES.sort_values(['status', 'category', 'admissions_all_ages'], ascending=[False, True, False])
CODES_OUT = CODES.copy()
for c in ['admissions_all_ages', 'admissions_aged_28d_to_17y']:
    CODES_OUT[c] = CODES_OUT[c].map(sup)
save(CODES_OUT, '02_dx_codes.csv')
print(CODES_OUT[CODES_OUT.status.ne('exclude') | CODES.admissions_all_ages.gt(0)].head(60).to_string(index=False))

## Laboratory items
Candidate items for the 22 blood tests and 5 validators are searched in the laboratory dictionary (Chinese and English labels). An
item is selected automatically when its label matches, its specimen is blood (or not recorded) and it has results; every candidate
is listed in `05_lab_items.csv` with its units and distribution, and `LAB_ITEMID_OVERRIDE` replaces a selection. Values are converted
to the Wuhan unit where the unit label allows it; a result without a unit label takes the item's usual unit. Censored results
(for example "<0.05") enter with the reporting limit, as in Wuhan. Results filed without an admission number are linked to the
admission when they lie within 24 h before it or during it (Wuhan linked emergency-department results the same way).

In [ ]:
LAB_CONCEPTS = {  # concept: (label pattern, exclusion pattern)
    'neutrophils_abs': (r'中性粒细胞.{0,4}(绝对|计数|数目|#)|中性粒细胞数|neutrophil.{0,12}(count|abs|#)|NEUT#|NE#|\bANC\b',
                        r'百分|%|比例|比率|percent|ratio|脑脊液|CSF|尿|胸|腹|浆膜'),
    'wbc': (r'白细胞计数|白细胞总数|白细胞数|^\s*白细胞\s*$|white blood cell|leukocyte count|\bWBC\b',
            r'尿|urine|脑脊液|CSF|粪|stool|介素|分类|胸|腹|浆膜|阴道|前列腺|精液'),
    'lymphocytes_pct': (r'淋巴细胞.{0,4}(百分|比例|比率|%)|lymphocyte.{0,12}(%|percent)|LYM%|LY%|LYMPH%',
                        r'绝对|#|计数|count|异型|atypical|脑脊液|CSF|亚群|CD\d'),
    'platelets': (r'血小板计数|^\s*血小板\s*$|血小板数|platelet count|^\s*platelets?\s*$|\bPLT\b',
                  r'压积|体积|分布宽度|MPV|PDW|PCT|聚集|抗体|大型|比率|功能'),
    'hemoglobin': (r'血红蛋白|hemoglobin|haemoglobin|\bHGB\b|\bHb\b',
                   r'糖化|A1c|平均|MCH|尿|胎儿|HbF|电泳|异常|游离|碳氧|高铁|还原|氧合|COHb|MetHb|O2Hb|HHb|变性'),
    'hs_crp': (r'C.?反应蛋白|C-?reactive|\bCRP\b|\bhsCRP\b|hs-CRP', r'脑脊液|CSF'),
    'procalcitonin': (r'降钙素原|procalcitonin|\bPCT\b', r'血小板|压积|platelet'),
    'alt': (r'丙氨酸.{0,3}氨基转移酶|谷丙转氨酶|\bALT\b|\bGPT\b|alanine amino', r'脑脊液|CSF'),
    'ast': (r'天[门冬].{0,4}氨酸.{0,3}氨基转移酶|谷草转氨酶|\bAST\b|\bGOT\b|aspartate amino', r'线粒体|mAST|同工酶|ASTm|脑脊液|CSF'),
    'ldh': (r'乳酸脱氢酶|lactate dehydrogenase|\bLDH?\b', r'同工酶|isoenzyme|羟丁酸|HBDH|脑脊液|CSF|胸|腹'),
    'ck': (r'肌酸激酶(?!同工)|creatine kinase|\bCK\b|\bCPK\b', r'同工酶|MB|isoenzyme|质量|mass|BB|MM'),
    'bilirubin_total': (r'总胆红素|total bilirubin|\bTBIL\b|\bT-?BIL\b', r'尿|脑脊液|经皮|transcutaneous'),
    'albumin': (r'白蛋白|albumin|\bALB\b', r'前白蛋白|prealbumin|尿|微量|球蛋白比|A/G|脑脊液|CSF|糖化|比值|缺血修饰'),
    'creatinine': (r'肌酐|creatinine|\bCREA?\b', r'尿|urine|清除|clearance|胱抑素|比值|ratio'),
    'urea': (r'尿素氮|血尿素|^\s*尿素|urea|\bBUN\b', r'尿液|尿尿素|24|urine|清除|clearance|比值|ratio|酶|呼气|breath'),
    'sodium': (r'钠(?!尿肽)|sodium|^\s*Na\+?\s*$|\bNa\b', r'尿|urine|钠尿肽|natriuretic|脑脊液|CSF|排泄'),
    'ammonia': (r'血氨|^\s*氨\s*$|ammonia|\bNH3\b', r'氨基|氨酸|amino|转移酶|尿|氨苄|呼气'),
    'lactate': (r'乳酸(?!脱氢酶)|lactate|lactic', r'脱氢酶|dehydrogenase|LDH|脑脊液|CSF'),
    'inr': (r'\bINR\b|国际标准化', None),
    'aptt': (r'活化部分凝血活酶|部分凝血活酶时间|\bAPTT\b|\baPTT\b|activated partial thromboplastin', r'比值|比率|ratio|纠正|对照|control'),
    'fibrinogen': (r'纤维蛋白原|fibrinogen|\bFIB\b|\bFbg\b', r'降解|degradation|FDP'),
    'd_dimer': (r'D.?二聚体|D-?dimer|\bD-?D\b', None),
    'il6': (r'白细胞介素.?6(?!\d)|白介素.?6(?!\d)|interleukin.?6(?!\d)|\bIL-?6(?!\d)', r'受体|receptor|IL-?6R'),
    'il10': (r'白细胞介素.?10|白介素.?10|interleukin.?10|\bIL-?10\b', r'受体|receptor'),
    'tnf_alpha': (r'肿瘤坏死因子|TNF', r'受体|receptor|TNFR'),
    'ifn_gamma': (r'干扰素|interferon|IFN', r'释放|release|IGRA|结核|T-SPOT|\bTB\b|[αβ]|alpha|beta'),
    'ferritin': (r'铁蛋白|ferritin', None),
}
BLOOD = r'blood|血|serum|plasma|whole|动脉|静脉|末梢|arterial|venous|capillary'
NOT_BLOOD = r'尿|urine|脑脊|CSF|cerebrospinal|胸水|腹水|pleural|ascit|粪|stool|痰|sputum|浆膜|joint|关节|透析'


def norm_unit(u):
    if not isinstance(u, str) or not u.strip():
        return ''
    u = u.strip().lower().replace('μ', 'u').replace('µ', 'u').replace('×', '').replace(' ', '').replace('（', '(').replace('）', ')')
    u = re.sub(r'^(x|\*)', '', u)
    u = re.sub(r'10(\^|\*|e|~)?9/l|g/l\(10\^9\)|10\^9l', '10^9/l', u) if re.search(r'10(\^|\*|e|~)?9', u) else u
    u = u.replace('iu/l', 'u/l').replace('sec', 's').replace('秒', 's').replace('meq/l', 'mmol/l')
    return u


# Conversion of each normalized unit to the Wuhan unit (factor); units not listed are not converted and are flagged.
CONV = {
    'neutrophils_abs': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1}, 'wbc': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1},
    'platelets': {'10^9/l': 1, '10^3/ul': 1, '/nl': 1, 'g/l': 1}, 'lymphocytes_pct': {'%': 1},
    'hemoglobin': {'g/l': 1, 'g/dl': 10}, 'hs_crp': {'mg/l': 1, 'mg/dl': 10}, 'procalcitonin': {'ng/ml': 1, 'ug/l': 1},
    'alt': {'u/l': 1}, 'ast': {'u/l': 1}, 'ldh': {'u/l': 1}, 'ck': {'u/l': 1},
    'bilirubin_total': {'umol/l': 1, 'mg/dl': 17.1}, 'albumin': {'g/l': 1, 'g/dl': 10}, 'creatinine': {'umol/l': 1, 'mg/dl': 88.42},
    'urea': {'mmol/l': 1}, 'sodium': {'mmol/l': 1}, 'ammonia': {'umol/l': 1, 'ug/dl': 0.587}, 'lactate': {'mmol/l': 1, 'mg/dl': 0.111},
    'inr': {'': 1, 'ratio': 1, 'inr': 1}, 'aptt': {'s': 1}, 'fibrinogen': {'g/l': 1, 'mg/dl': 0.01},
    'd_dimer': {'mg/lfeu': 1, 'mg/l': 1, 'ug/ml': 1, 'ug/mlfeu': 1, 'ug/l': 0.001, 'ng/ml': 0.001, 'ug/lfeu': 0.001, 'ng/mlfeu': 0.001},
    'il6': {'pg/ml': 1, 'ng/l': 1}, 'il10': {'pg/ml': 1, 'ng/l': 1}, 'tnf_alpha': {'pg/ml': 1, 'ng/l': 1}, 'ifn_gamma': {'pg/ml': 1, 'ng/l': 1},
    'ferritin': {'ng/ml': 1, 'ug/l': 1},
}
DLAB = read('D_LABITEMS')
C_ITEM = pick(DLAB.columns, 'ITEMID')
C_LCN = pick(DLAB.columns, 'LABEL_CN', contains=['LABEL', 'CN'])
C_LEN = pick([c for c in DLAB.columns if c != C_LCN], 'LABEL', contains=['LABEL'])
C_FLUID = pick(DLAB.columns, 'FLUID', 'SPECIMEN')
C_LOINC = pick(DLAB.columns, 'LOINC_CODE', contains=['LOINC'])
DLAB = pd.DataFrame({'ITEMID': DLAB[C_ITEM].astype(str), 'LABEL_CN': DLAB[C_LCN] if C_LCN else np.nan, 'LABEL': DLAB[C_LEN] if C_LEN else np.nan,
                     'FLUID': DLAB[C_FLUID] if C_FLUID else np.nan, 'LOINC_CODE': DLAB[C_LOINC] if C_LOINC else np.nan}).drop_duplicates('ITEMID')
DLAB['TXT'] = (DLAB.LABEL.fillna('') + ' | ' + DLAB.LABEL_CN.fillna('')).str.strip()
def label_match(df, inc, exc):
    """Rows whose Chinese or English label matches inc and neither label matches exc (each label tested on its own)."""
    labs = [df[c].fillna('').astype(str) for c in ('LABEL_CN', 'LABEL')]
    m = labs[0].str.contains(inc, flags=re.I, regex=True) | labs[1].str.contains(inc, flags=re.I, regex=True)
    if exc:
        m &= ~(labs[0].str.contains(exc, flags=re.I, regex=True) | labs[1].str.contains(exc, flags=re.I, regex=True))
    return df[m]


cand = []
for concept, (inc, exc) in LAB_CONCEPTS.items():
    for r in label_match(DLAB, inc, exc).itertuples():
        fl = r.FLUID if isinstance(r.FLUID, str) else ''
        blood = (not fl.strip()) or (bool(re.search(BLOOD, fl, flags=re.I)) and not re.search(NOT_BLOOD, fl, flags=re.I))
        cand.append({'concept': concept, 'ITEMID': r.ITEMID, 'LABEL': r.LABEL, 'LABEL_CN': r.LABEL_CN, 'FLUID': r.FLUID,
                     'LOINC_CODE': r.LOINC_CODE, 'blood': blood})
LCAND = pd.DataFrame(cand, columns=['concept', 'ITEMID', 'LABEL', 'LABEL_CN', 'FLUID', 'LOINC_CODE', 'blood'])
CAND_IDS = set(LCAND.ITEMID)
print('Candidate laboratory items:', LCAND.groupby('concept').size().reindex(LAB22 + VALIDATORS).fillna(0).astype(int).to_dict())

# Admissions whose laboratory results are needed: every admission of any candidate definition, aged 28 days to 17 years.
CAND_ADM = (ADM_INC | ADM_ENC | ADM_TXT_INC) & _age_ok
CAND_SUBJ = set(ADM.loc[sorted(CAND_ADM), 'SUBJECT_ID']) if CAND_ADM else set()
NUM = re.compile(r'^\s*(<=|>=|<|>|≤|≥)?\s*([-+]?\d+(?:\.\d+)?)\s*$')
unit_counts, item_rows, item_hadm, keep = {}, {}, {}, []
for ch in read('LABEVENTS', ['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM'], chunksize=CHUNK):
    ch = ch[ch.ITEMID.isin(CAND_IDS)]
    if ch.empty:
        continue
    for (it, u), k in ch.groupby(['ITEMID', ch.VALUEUOM.fillna('')]).size().items():
        unit_counts.setdefault(it, {}).setdefault(u, 0)
        unit_counts[it][u] += int(k)
    for it, k in ch.ITEMID.value_counts().items():
        item_rows[it] = item_rows.get(it, 0) + int(k)
    for it, g in ch.groupby('ITEMID'):
        item_hadm.setdefault(it, set()).update(g.HADM_ID.dropna().unique())
    sel = ch.HADM_ID.isin(CAND_ADM) | (ch.HADM_ID.isna() & ch.SUBJECT_ID.isin(CAND_SUBJ))
    if sel.any():
        keep.append(ch[sel])
LAB = pd.concat(keep, ignore_index=True) if keep else pd.DataFrame(columns=['SUBJECT_ID', 'HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM', 'VALUEUOM'])
LAB['T'] = to_dt(LAB.CHARTTIME)
LAB['V'] = pd.to_numeric(LAB.VALUENUM, errors='coerce')
_p = LAB.VALUE.astype(str).str.extract(NUM)
LAB['CENSORED'] = LAB.V.isna() & _p[0].notna()
LAB['V'] = LAB.V.fillna(pd.to_numeric(_p[1], errors='coerce'))
# usual unit of each item (most frequent non-empty label) for results without a unit label
ITEM_UNIT = {it: max(((u, k) for u, k in d.items() if u), key=lambda x: x[1], default=('', 0))[0] for it, d in unit_counts.items()}
LAB['UNIT_RAW'] = LAB.VALUEUOM.fillna('')
LAB['UNIT_N'] = [norm_unit(u if u else ITEM_UNIT.get(it, '')) for u, it in zip(LAB.UNIT_RAW, LAB.ITEMID)]
# link results without an admission number to the subject's candidate admission (within 24 h before admission or during it)
_noh = LAB.HADM_ID.isna()
if _noh.any():
    _c = ADM.loc[sorted(CAND_ADM), ['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME']].reset_index(drop=True)
    _l = LAB[_noh].reset_index().merge(_c, on='SUBJECT_ID', how='inner', suffixes=('', '_A'))
    _l = _l[(_l['T'] >= _l.ADMITTIME - pd.Timedelta(hours=24)) & (_l['T'] <= _l.DISCHTIME)]
    _l = _l.sort_values(['index', 'ADMITTIME']).drop_duplicates('index')
    LAB.loc[_l['index'].values, 'HADM_ID'] = _l['HADM_ID_A'].values
    LAB['LINKED'] = False
    LAB.loc[_l['index'].values, 'LINKED'] = True
else:
    LAB['LINKED'] = False
LAB = LAB[LAB.HADM_ID.isin(CAND_ADM) & LAB.V.notna() & LAB['T'].notna()].copy()
print(f'Laboratory rows kept for candidate admissions: {len(LAB)} (linked without an admission number: {int(LAB.LINKED.sum())})')

In [ ]:
# Item statistics, automatic selection and conversion to the Wuhan unit.
def item_stats(it):
    d = LAB[LAB.ITEMID.eq(it)]
    units = '; '.join(f'{u or "(none)"} ({sup(k)})' for u, k in sorted(unit_counts.get(it, {}).items(), key=lambda kv: -kv[1])[:4])
    return {'rows_all': item_rows.get(it, 0), 'admissions_all': len(item_hadm.get(it, set())), 'rows_candidate_admissions': len(d),
            'units': units, 'pct_censored': pct(int(d.CENSORED.sum()), len(d)), 'median_candidate_admissions': q(d.V)}


LCAND = pd.concat([LCAND.reset_index(drop=True), pd.DataFrame([item_stats(i) for i in LCAND.ITEMID])], axis=1)


def auto_select(concept):
    g = LCAND[LCAND.concept.eq(concept) & LCAND.blood & LCAND.rows_candidate_admissions.gt(0)]
    if g.empty:
        g = LCAND[LCAND.concept.eq(concept) & LCAND.rows_candidate_admissions.gt(0)]
    return sorted(g.ITEMID.unique().tolist())


LAB_ITEMS = {c: [str(i) for i in LAB_ITEMID_OVERRIDE.get(c, auto_select(c))] for c in LAB22 + VALIDATORS}
LCAND['selected'] = [it in LAB_ITEMS.get(c, []) for c, it in zip(LCAND.concept, LCAND.ITEMID)]
_out = LCAND.copy()
for c in ['rows_all', 'admissions_all', 'rows_candidate_admissions']:
    _out[c] = _out[c].map(sup)
save(_out.drop(columns=['blood']).assign(blood_specimen=LCAND.blood), '05_lab_items.csv')
print('Selected laboratory items:', json.dumps(LAB_ITEMS, ensure_ascii=False))

_rows = []
for c, items in LAB_ITEMS.items():
    if items:
        d = LAB[LAB.ITEMID.isin(items)].copy()
        d['concept'] = c
        d['FACTOR'] = d.UNIT_N.map(CONV.get(c, {}))
        _rows.append(d)
LABC = pd.concat(_rows, ignore_index=True) if _rows else pd.DataFrame(columns=list(LAB.columns) + ['concept', 'FACTOR'])
LABC['VC'] = LABC.V * LABC.FACTOR                                   # value in the Wuhan unit (NaN when the unit is not convertible)
UNIT_USE = (LABC.groupby(['concept', 'UNIT_N']).size().rename('rows').reset_index()
            .assign(factor=lambda d: [CONV.get(c, {}).get(u) for c, u in zip(d.concept, d.UNIT_N)]))

## Vital signs
Temperature, heart rate (monitor heart rate where charted, otherwise the pulse) and respiratory rate: the first charted value within
24 h after admission, as in Wuhan.

In [ ]:
DIT = read('D_ITEMS')
C_IID = pick(DIT.columns, 'ITEMID')
C_ICN = pick(DIT.columns, 'LABEL_CN', contains=['LABEL', 'CN'])
C_IEN = pick([c for c in DIT.columns if c != C_ICN], 'LABEL', contains=['LABEL'])
DIT = pd.DataFrame({'ITEMID': DIT[C_IID].astype(str), 'LABEL_CN': DIT[C_ICN] if C_ICN else np.nan, 'LABEL': DIT[C_IEN] if C_IEN else np.nan,
                    'UNITNAME': DIT['UNITNAME'] if 'UNITNAME' in DIT else np.nan, 'LINKSTO': DIT['LINKSTO'] if 'LINKSTO' in DIT else np.nan})
DIT['TXT'] = (DIT.LABEL.fillna('') + ' | ' + DIT.LABEL_CN.fillna('')).str.strip()
VITAL_CONCEPTS = {
    'temp_c': (r'体温|temperature', r'皮肤|skin|室温|环境|room'), 'heart_rate': (r'心率|heart rate|\bHR\b', r'胎'),
    'pulse': (r'脉搏|pulse', r'血氧|oximetry|压|pressure'),
    'resp_rate': (r'呼吸频率|呼吸次数|^\s*呼吸\s*$|respiratory rate|resp(iration)? rate|^\s*respiration\s*$|\bRR\b',
                  r'呼吸机|ventilat|末|设定|set|自主|spont'),
}
vc = []
for concept, (inc, exc) in VITAL_CONCEPTS.items():
    for r in label_match(DIT, inc, exc).itertuples():
        vc.append({'concept': concept, 'ITEMID': r.ITEMID, 'LABEL': r.LABEL, 'LABEL_CN': r.LABEL_CN, 'UNITNAME': r.UNITNAME, 'LINKSTO': r.LINKSTO})
VCAND = pd.DataFrame(vc, columns=['concept', 'ITEMID', 'LABEL', 'LABEL_CN', 'UNITNAME', 'LINKSTO'])
VIDS = set(VCAND.ITEMID)
vrows, vkeep = {}, []
for ch in read('CHARTEVENTS', ['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM'], chunksize=CHUNK):
    ch = ch[ch.ITEMID.isin(VIDS)]
    if ch.empty:
        continue
    for it, k in ch.ITEMID.value_counts().items():
        vrows[it] = vrows.get(it, 0) + int(k)
    ch = ch[ch.HADM_ID.isin(CAND_ADM)]
    if not ch.empty:
        vkeep.append(ch)
VIT = pd.concat(vkeep, ignore_index=True) if vkeep else pd.DataFrame(columns=['HADM_ID', 'ITEMID', 'CHARTTIME', 'VALUE', 'VALUENUM'])
VIT['T'] = to_dt(VIT.CHARTTIME)
VIT['V'] = pd.to_numeric(VIT.VALUENUM, errors='coerce').fillna(pd.to_numeric(VIT.VALUE, errors='coerce'))
VIT = VIT[VIT.V.notna() & VIT['T'].notna()]
VCAND['rows_all'] = VCAND.ITEMID.map(vrows).fillna(0).astype(int)
VCAND['rows_candidate_admissions'] = VCAND.ITEMID.map(VIT.ITEMID.value_counts()).fillna(0).astype(int)
VCAND['median_candidate_admissions'] = [q(VIT.loc[VIT.ITEMID.eq(i), 'V']) for i in VCAND.ITEMID]
VITAL_ITEMS = {c: [str(i) for i in VITAL_ITEMID_OVERRIDE.get(c, sorted(VCAND[VCAND.concept.eq(c) & VCAND.rows_candidate_admissions.gt(0)].ITEMID))]
               for c in VITAL_CONCEPTS}
VCAND['selected'] = [it in VITAL_ITEMS.get(c, []) for c, it in zip(VCAND.concept, VCAND.ITEMID)]
_v = VCAND.copy()
for c in ['rows_all', 'rows_candidate_admissions']:
    _v[c] = _v[c].map(sup)
save(_v, '06_vital_items.csv')
print('Selected vital-sign items:', json.dumps(VITAL_ITEMS, ensure_ascii=False))

In [ ]:
# First value of every indicator per admission, for two anchors: hospital admission (primary) and first ICU entry (alternative).
ANCHORS = {'admission': ADM.ADMITTIME, 'icu_entry': ADM.ICU_IN}


def first_values(anchor):
    a = ANCHORS[anchor]
    res = {}
    if len(LABC):
        d = LABC[LABC.VC.notna()].copy()
        d['H'] = (d['T'] - d.HADM_ID.map(a)).dt.total_seconds() / 3600
        for c in LAB22 + VALIDATORS:
            lo, hi = VALIDATOR_WINDOW_H if c in VALIDATORS else LAB_WINDOW_H
            w = d[d.concept.eq(c) & d.H.between(lo, hi)].sort_values(['HADM_ID', 'T'])
            res[c] = w.drop_duplicates('HADM_ID').set_index('HADM_ID').VC
    if len(VIT):
        v = VIT.copy()
        v['H'] = (v['T'] - v.HADM_ID.map(a)).dt.total_seconds() / 3600
        v = v[v.H.between(*VITAL_WINDOW_H)].sort_values(['HADM_ID', 'T'])
        for c in ['temp_c', 'heart_rate', 'pulse', 'resp_rate']:
            res[c] = v[v.ITEMID.isin(VITAL_ITEMS.get(c, []))].drop_duplicates('HADM_ID').set_index('HADM_ID').V
        hr = res.get('heart_rate', pd.Series(dtype=float))
        res['heart_rate'] = hr.combine_first(res.get('pulse', pd.Series(dtype=float)))
        t = res.get('temp_c', pd.Series(dtype=float))
        res['temp_c'] = t.where(t < 45, (t - 32) / 1.8)              # Fahrenheit, if any, to Celsius
    F = pd.DataFrame(res).reindex(index=sorted(CAND_ADM))
    for c in LAB22 + VALIDATORS + ['temp_c', 'heart_rate', 'resp_rate']:
        if c not in F:
            F[c] = np.nan
    return F


FV = {k: first_values(k) for k in ANCHORS}
NTESTS = {k: FV[k][LAB22].notna().sum(axis=1) for k in ANCHORS}

## Cohort flow and the registered threshold
Primary definition, in the Wuhan order: an included acute encephalopathy code in the discharge diagnoses; the child's first such
admission (index admission); aged 28 days to 17 years at it; at least 11 of the 22 blood tests within 24 h before to 24 h after
admission (the Wuhan clustering population). Alternatives, for information only: encephalitis codes added; the admission diagnosis added; first ICU entry as the
time anchor instead of hospital admission.

In [ ]:
DEFS = {'primary': (ADM_INC, 'admission'), 'with_encephalitis': (ADM_INC | ADM_ENC, 'admission'),
        'with_admission_diagnosis': (ADM_INC | ADM_TXT_INC, 'admission'), 'icu_entry_anchor': (ADM_INC, 'icu_entry')}


def flow(hadm, anchor):
    steps = []
    A = ADM.loc[sorted(hadm & set(ADM.index))]
    steps.append(('Admissions with a qualifying diagnosis', len(A), A.SUBJECT_ID.nunique()))
    A = A.sort_values(['SUBJECT_ID', 'ADMITTIME', 'HADM_ID']).drop_duplicates('SUBJECT_ID')
    steps.append(("First such admission per child (index admission)", len(A), A.SUBJECT_ID.nunique()))
    A = A[A.AGE_DAYS.ge(AGE_MIN_DAYS)]
    steps.append(('Aged 28 days or older at the index admission', len(A), A.SUBJECT_ID.nunique()))
    A = A[A.AGE_DAYS.lt(AGE_MAX_YEARS * 365.25)]
    steps.append(('Younger than 18 years', len(A), A.SUBJECT_ID.nunique()))
    if anchor == 'icu_entry':
        A = A[A.ICU_IN.notna()]
        steps.append(('With an ICU stay (time anchor: first ICU entry)', len(A), A.SUBJECT_ID.nunique()))
    nt = NTESTS[anchor].reindex(A.index).fillna(0)
    A = A[nt.ge(MIN_TESTS)]
    steps.append((f'At least {MIN_TESTS} of the 22 blood tests in the window (eligible)', len(A), A.SUBJECT_ID.nunique()))
    return A, steps


COH, FLOW_ROWS = {}, []
for name, (hadm, anchor) in DEFS.items():
    COH[name], steps = flow(hadm, anchor)
    for i, (s, na, nc) in enumerate(steps, 1):
        FLOW_ROWS.append({'definition': name, 'step': i, 'description': s, 'admissions': sup(na), 'children': sup(nc)})
save(pd.DataFrame(FLOW_ROWS), '03_cohort_flow.csv')
print(pd.DataFrame(FLOW_ROWS).to_string(index=False))
N_ELIG = len(COH['primary'])
GATE = 'meets' if N_ELIG >= GATE_N else 'does not meet'
print(f'\nEligible children (primary definition): {N_ELIG}. This {GATE} the registered threshold of {GATE_N}.')

In [ ]:
# Profile of each cohort (aggregate), category mix, and missingness of the 24 clustering indicators. The only outcome counted is the
# overall number of in-hospital deaths.
def profile(A, anchor):
    n = len(A)
    F = FV[anchor].reindex(A.index)
    k_died = int(A.DIED.sum())
    k_male = int(A.GENDER.astype(str).str.upper().str[0].eq('M').sum())
    rows = {'children': sup(n), 'age_years_median_iqr': q(A.AGE_DAYS / 365.25), 'male_pct': pct(k_male, n),
            'hospital_los_days_median_iqr': q(A.LOS_DAYS), 'in_hospital_deaths': sup(k_died), 'in_hospital_deaths_pct': pct(k_died, n),
            'blood_tests_of_22_median_iqr': q(NTESTS[anchor].reindex(A.index)),
            'missing_pct_24_indicators': round(100 * float(F[MODEL24].isna().values.mean()), 1) if n else None,
            'children_with_icu_stay': sup(int(A.ICU_IN.notna().sum()))}
    for u, k in A.FIRST_UNIT.fillna('(no ICU stay)').value_counts().items():
        rows[f'first_icu_unit: {u}'] = sup(int(k))
    return rows


PROF = pd.DataFrame({name: profile(COH[name], DEFS[name][1]) for name in DEFS})
PROF.index.name = 'measure'
PROF.reset_index().to_csv(OUT / '04_cohort_profile.csv', index=False, encoding='utf-8-sig')
print(PROF.to_string())
cat_rows = []
for name in DEFS:
    A = COH[name]
    cats = A.index.to_series().map(ADM_CAT).fillna('Encephalitis or CNS infection only')
    for c, k in cats.value_counts().items():
        cat_rows.append({'definition': name, 'category': c, 'children': sup(int(k))})
    co = A.index.to_series().map(ADM_EXCL).dropna()
    for c, k in pd.Series([x for lst in co for x in lst]).value_counts().items():
        cat_rows.append({'definition': name, 'category': f'also coded: {c}', 'children': sup(int(k))})
save(pd.DataFrame(cat_rows), '04b_category_mix.csv')

## Indicator availability and unit check (primary cohort)
For each indicator: the share of eligible children with a value in the window, and its distribution in the Wuhan unit, next to
Wuhan. A median more than 3-fold different from Wuhan (10-fold for the strongly skewed markers) is flagged: it points to a unit or
item problem, although case mix also moves the medians (PIC children were all in intensive care).

In [ ]:
SKEWED = {'hs_crp', 'procalcitonin', 'alt', 'ast', 'ldh', 'ck', 'd_dimer', 'ammonia', 'il6', 'il10', 'tnf_alpha', 'ifn_gamma', 'ferritin'}
A = COH['primary']
F = FV['admission'].reindex(A.index)
av = []
for c in LAB22 + ['temp_c', 'heart_rate', 'resp_rate'] + VALIDATORS:
    s = F[c] if c in F else pd.Series(dtype=float)
    k = int(s.notna().sum())
    w = WUHAN.get(c, {})
    med = float(s.median()) if k >= 5 else None
    ratio = round(med / w['median'], 2) if med is not None and w.get('median') else None
    lim = 10 if c in SKEWED else 3
    flag = '' if ratio is None or 1 / lim <= ratio <= lim else f'median ratio outside 1/{lim}-{lim}'
    if c in LAB22 + VALIDATORS:
        uu = UNIT_USE[UNIT_USE.concept.eq(c)]
        bad = uu[uu.factor.isna()]
        units = '; '.join(f'{u or "(none)"}: {sup(r)}' for u, r in zip(uu.UNIT_N, uu.rows))
        if len(bad):
            flag = (flag + '; ' if flag else '') + 'unconvertible unit: ' + ', '.join(u or '(none)' for u in bad.UNIT_N)
        items = ', '.join(LAB_ITEMS.get(c, [])) or 'none found'
    else:
        units = '' if c != 'heart_rate' else 'monitor heart rate, else pulse'
        items = ', '.join(VITAL_ITEMS.get(c, []) + (VITAL_ITEMS.get('pulse', []) if c == 'heart_rate' else [])) or 'none found'
    av.append({'indicator': LABEL[c], 'concept': c, 'group': 'blood test' if c in LAB22 else ('vital sign' if c in ('temp_c', 'heart_rate', 'resp_rate') else 'validator'),
               'in_model': c in MODEL24, 'pic_items': items, 'pic_units_rows': units, 'wuhan_unit': w.get('unit', ''),
               'pic_measured': sup(k), 'pic_pct': pct(k, len(A)), 'wuhan_pct': w.get('pct_measured_clustered'),
               'pic_median_iqr': q(s), 'wuhan_median_iqr': f"{w['median']:.3g} ({w['p25']:.3g}-{w['p75']:.3g})" if w else None,
               'median_ratio_pic_to_wuhan': ratio, 'flag': flag})
AV = save(pd.DataFrame(av), '07_indicator_availability.csv')
print(AV[['indicator', 'pic_items', 'pic_measured', 'pic_pct', 'wuhan_pct', 'pic_median_iqr', 'wuhan_median_iqr', 'flag']].to_string(index=False))

## Consciousness at admission (optional table)
PIC has no Glasgow coma scale. `EMR_SYMPTOMS` holds symptoms extracted from the notes; this table counts eligible children with a
consciousness-related symptom, to judge whether the registered prognostic test (subphenotype beyond age and consciousness) can be
approximated in PIC. Only counts and attribute values are reported.

In [ ]:
CONS_TERMS = {'coma': r'昏迷|coma', 'stupor': r'昏睡|stupor', 'drowsiness': r'嗜睡|somnolen|drows|letharg',
              'altered consciousness': r'意识障碍|意识不清|神志不清|意识模糊|谵妄|confus|delirium|unconscious|consciousness'}
cons_rows = []
if 'EMR_SYMPTOMS' in FILES:
    ES = read('EMR_SYMPTOMS', ['HADM_ID', 'RECORDTIME', 'SYMPTOM_NAME_CN', 'SYMPTOM_NAME', 'SYMPTOM_ATTRIBUTE'])
    ES = ES[ES.HADM_ID.isin(set(A.index))].copy()
    if ES.empty:
        ES = pd.DataFrame(columns=['HADM_ID', 'RECORDTIME', 'SYMPTOM_NAME_CN', 'SYMPTOM_NAME', 'SYMPTOM_ATTRIBUTE'])
    ES['TXT'] = ES.reindex(columns=['SYMPTOM_NAME_CN', 'SYMPTOM_NAME']).fillna('').astype(str).agg(' | '.join, axis=1)
    ES['H'] = (to_dt(ES.RECORDTIME) - ES.HADM_ID.map(ADM.ADMITTIME)).dt.total_seconds() / 3600
    for term, pat in CONS_TERMS.items():
        e = ES[ES.TXT.str.contains(pat, flags=re.I, regex=True)]
        attrs = e.SYMPTOM_ATTRIBUTE.fillna('(none)').value_counts().head(6) if 'SYMPTOM_ATTRIBUTE' in e else pd.Series(dtype=int)
        cons_rows.append({'term': term, 'children_with_record': sup(e.HADM_ID.nunique()),
                          'children_with_record_within_24h': sup(e[e.H.between(-24, 24)].HADM_ID.nunique()),
                          'attribute_values': '; '.join(f'{a}: {sup(k)}' for a, k in attrs.items())})
else:
    cons_rows.append({'term': 'EMR_SYMPTOMS not found', 'children_with_record': None, 'children_with_record_within_24h': None, 'attribute_values': ''})
CONS = save(pd.DataFrame(cons_rows), '08_consciousness_terms.csv')
print(CONS.to_string(index=False))

In [ ]:
# summary.md: the numbers to paste back.
def t(x):
    return '' if x is None or (not isinstance(x, str) and pd.isna(x)) else str(x)


lines = []
if FIXTURE:
    lines += ['**TEST RUN ON A SYNTHETIC FIXTURE: every number is meaningless and must never be reported.**', '']
lines += ['# PIC_SUB_01 summary (aggregate; counts 1-4 shown as <5)', '',
          f'Run {time.strftime("%d %b %Y %H:%M")}, {round((time.time() - T0) / 60, 1)} min; python {platform.python_version()}, pandas {pd.__version__}.', '',
          '## Registered threshold', '',
          f'Eligible children, primary definition: **{sup(N_ELIG)}**; this **{GATE}** the registered threshold of {GATE_N} (analysis plan, decision P15).', '',
          '## Cohort flow', '', '| Definition | Step | Admissions | Children |', '|---|---|---|---|']
lines += [f"| {r['definition']} | {r['description']} | {t(r['admissions'])} | {t(r['children'])} |" for r in FLOW_ROWS]
lines += ['', '## Eligible children', '', '| Measure | ' + ' | '.join(DEFS) + ' |', '|---' * (len(DEFS) + 1) + '|']
lines += [f'| {m} | ' + ' | '.join(t(v) for v in PROF.loc[m]) + ' |' for m in PROF.index]
lines += ['', '## Category mix (primary definition)', '', '| Category | Children |', '|---|---|']
lines += [f"| {r['category']} | {t(r['children'])} |" for r in cat_rows if r['definition'] == 'primary']
lines += ['', '## Indicators (primary definition)', '',
          '| Indicator | In model | PIC items | PIC measured | PIC % | Wuhan % | PIC median (IQR) | Wuhan median (IQR) | Flag |',
          '|---|---|---|---|---|---|---|---|---|']
lines += [f"| {r.indicator} | {'yes' if r.in_model else ''} | {t(r.pic_items)} | {t(r.pic_measured)} | {t(r.pic_pct)} | "
          f"{t(r.wuhan_pct)} | {t(r.pic_median_iqr)} | {t(r.wuhan_median_iqr)} | {t(r.flag)} |" for r in AV.itertuples()]
lines += ['', '## Consciousness terms in EMR_SYMPTOMS (primary definition)', '', '| Term | Children | Within 24 h | Attribute values |', '|---|---|---|---|']
lines += [f"| {r['term']} | {t(r['children_with_record'])} | {t(r['children_with_record_within_24h'])} | {t(r['attribute_values'])} |" for r in cons_rows]
n_codes_inc = int((CODES.status.eq('include') & CODES.admissions_all_ages.gt(0)).sum())
lines += ['', '## Files for review', '',
          f'* `02_dx_codes.csv`: {len(CODES)} classified diagnosis codes ({n_codes_inc} included codes in use), with titles, status and counts.',
          '* `05_lab_items.csv` and `06_vital_items.csv`: candidate items, selection, units and distributions.',
          '* `03_cohort_flow.csv`, `04_cohort_profile.csv`, `04b_category_mix.csv`, `07_indicator_availability.csv`, `08_consciousness_terms.csv`.',
          '', 'Nothing in this run is computed by subphenotype; the model has not been applied.']
(OUT / 'summary.md').write_text('\n'.join(lines) + '\n', encoding='utf-8')
(OUT / 'run_info.json').write_text(json.dumps({'notebook': 'PIC_SUB_01_feasibility', 'synthetic_fixture': FIXTURE, 'python': platform.python_version(),
                                                'pandas': pd.__version__, 'numpy': np.__version__, 'minutes': round((time.time() - T0) / 60, 1),
                                                'parameters': {'AGE_MIN_DAYS': AGE_MIN_DAYS, 'AGE_MAX_YEARS': AGE_MAX_YEARS, 'LAB_WINDOW_H': LAB_WINDOW_H,
                                                               'VALIDATOR_WINDOW_H': VALIDATOR_WINDOW_H, 'VITAL_WINDOW_H': VITAL_WINDOW_H,
                                                               'MIN_TESTS': MIN_TESTS, 'GATE_N': GATE_N, 'DX_STATUS_OVERRIDE': DX_STATUS_OVERRIDE,
                                                               'LAB_ITEMID_OVERRIDE': LAB_ITEMID_OVERRIDE, 'VITAL_ITEMID_OVERRIDE': VITAL_ITEMID_OVERRIDE}},
                                               indent=1, ensure_ascii=False), encoding='utf-8')
print('\n'.join(lines[:12]))

In [ ]:
# Privacy check of every output file: no identifier columns, no dates, and no count from 1 to 4.
ID_COL = re.compile(r'SUBJECT|HADM|ICUSTAY|ROW_ID|PATIENT', flags=re.I)
DATE = re.compile(r'\b(19|20|21)\d{2}-\d{2}-\d{2}\b')
COUNT_COLS = {'rows', 'rows_all', 'admissions_all', 'rows_candidate_admissions', 'admissions', 'children', 'admissions_all_ages',
              'admissions_aged_28d_to_17y', 'pic_measured', 'children_with_record', 'children_with_record_within_24h'}
problems = []
for p in sorted(OUT.iterdir()):
    text = p.read_text(encoding='utf-8-sig')
    if p.name != 'run_info.json' and DATE.search(text):
        problems.append(f'{p.name}: date-like text')
    if p.suffix == '.csv':
        df = pd.read_csv(p, dtype=str, keep_default_na=False)
        problems += [f'{p.name}: identifier column {c}' for c in df.columns if ID_COL.search(c)]
        for c in set(df.columns) & COUNT_COLS:
            if df[c].isin(['1', '2', '3', '4']).any():
                problems.append(f'{p.name}: unsuppressed small count in {c}')
if problems:
    raise RuntimeError('Privacy check failed; do not share the outputs:\n' + '\n'.join(problems))
print(f'Privacy check passed for {len(list(OUT.iterdir()))} files in {OUT}. Send summary.md (or the folder).')